<a href="https://colab.research.google.com/github/botirbekzulf-sudo/ML/blob/main/%D0%A3%D1%80%D0%BE%D0%BA19_%D1%81%D0%B0%D0%BC%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚙️ Урок 19 — Самостоятельная работа: AutoML

### Как работать
Читай · запускай · выполняй задания 📝 · отвечай словами ✍️.

> 🎯 К концу урока ты запустишь перебор моделей (мини-AutoML) на пингвинах и сравнишь его с ручной моделью.

Блок 1 работает сразу — интернет не нужен.

## Шаг 1 · Мини-AutoML: пусть машина переберёт модели
Мы берём знакомых пингвинов и просим несколько моделей посоревноваться. Запусти — получишь лидерборд. Настоящий AutoML делает то же самое, только автоматически и с сотнями моделей.

In [2]:
import seaborn as sns, warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# пингвины — стандарт курса: убираем пропуски и колонку sex
df = sns.load_dataset('penguins').dropna().drop(columns=['sex'])
num = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
cat = ['island']
X = df[num+cat]; y = df['species']   # предсказываем вид пингвина

# подготовка: числа масштабируем, категории кодируем
prep = ColumnTransformer([
    ('n', StandardScaler(), num),
    ('c', OneHotEncoder(handle_unknown='ignore'), cat)])

# участники турнира
models = {'KNN':KNeighborsClassifier(),
          'Дерево':DecisionTreeClassifier(random_state=42),
          'ЛогРег':LogisticRegression(max_iter=1000),
          'Лес':RandomForestClassifier(random_state=42),
          'Бустинг':GradientBoostingClassifier(random_state=42)}

board = []
for name, m in models.items():
    # cross_val_score = честная проверка: 5 раз делит данные и усредняет
    score = cross_val_score(Pipeline([('prep',prep),('model',m)]), X, y, cv=5).mean()
    board.append((name, score))

print('=== Лидерборд (мини-AutoML) ===')
for name, s in sorted(board, key=lambda x:-x[1]):
    print(f'{name:8s} {s:.1%}')

=== Лидерборд (мини-AutoML) ===
ЛогРег   99.4%
Лес      99.1%
Бустинг  98.5%
KNN      98.2%
Дерево   97.9%


✍️ **Ответь.** Какая модель наверху лидерборда? Совпала ли она с той, что мы выбирали вручную (лес)?
Насколько отличается лучшая модель от худшей?

*Ответ:* наверху лидерборта ЛогРег 99.4%, нет лес на 2 месте хотя он тоже точный, отличие на 1.5%

### 📝 Задание — добавь модель
Добавь в словарь `models` ещё одну модель и снова построй лидерборд. Например `SVM`.

In [5]:
from sklearn.svm import SVC

# добавляем нового участника
models['SVM'] = SVC()

# повторяем турнир
board = []
for name, m in models.items():
    score = cross_val_score(Pipeline([('prep',prep),('model',m)]), X, y, cv=5).mean()
    board.append((name, score))

print('=== Лидерборд с новой моделью ===')
for name, s in sorted(board, key=lambda x:-x[1]):
    print(f'{name:8s} {s:.1%}')

=== Лидерборд с новой моделью ===
ЛогРег   99.4%
SVM      99.1%
Лес      99.1%
Бустинг  98.5%
KNN      98.2%
Дерево   97.9%


## Шаг 2 · AutoML — не волшебная кнопка
✍️ **Вопрос.** Представь, AutoML выдал точность **95%**. Что нужно проверить, прежде чем радоваться?

*Ответ:*

<details><summary>Подсказка</summary>Нет ли утечки данных, та ли метрика, честная ли проверка (test/CV).</details>

💡 **Заметь:** на пингвинах все модели дали ~98–99%. Разброс крошечный — потому что данные маленькие и чистые. На больших и грязных данных AutoML обычно выигрывает у ручной модели заметнее.

## ✅ Проверь себя
1. Что автоматизирует AutoML?
2. Что он **НЕ** заменяет?
3. Что такое Hugging Face Hub?

<details><summary>Ответы</summary>

1. Перебор моделей, настроек и ансамблей — выбирает лучшую.
2. Понимание качества: проверку утечки и выбор метрики.
3. Склад готовых обученных моделей — берёшь и используешь.
</details>

## 🏁 Финальное задание (в Colab, нужен интернет)
Запусти **настоящий** AutoGluon на пингвинах и сравни его лучшую модель со своей ручной.
Обрати внимание: label = `species`, и никакой ручной подготовки — AutoGluon делает всё сам.

In [ ]:
# Только в Colab с интернетом
!pip install autogluon.tabular -q
from autogluon.tabular import TabularPredictor
import seaborn as sns
from sklearn.model_selection import train_test_split

df = sns.load_dataset('penguins').dropna().drop(columns=['sex'])
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

predictor = TabularPredictor(label='species').fit(train_df, time_limit=120)
predictor.leaderboard(test_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.8 MB/s eta 0:00:00


No path specified. Models will be saved in: "AutogluonModels/ag-20260815_070742"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.09 GB / 12.67 GB (87.5%)
Disk Space Avail:   87.06 GB / 107.72 GB (80.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabular

---
### 🎉 Готово!
Ты увидел, как машина сама перебирает модели — и понял, что это не магия, а перебор, который ты уже умеешь делать руками. Дальше на уроке 20 превратим модель в настоящий сайт!